# SPINE-GPE v7 — PNAD COVID Certifier v1.0.2

Hardening da série maio–novembro/2020:

- corrige `C007` (`07`, `7.0`, `7` → `7`);
- recalcula `pandemic_delivery_self_employed`;
- audita cobertura e missingness de `C014`;
- cria outputs imutáveis por Run ID;
- promove o alias `latest` apenas após aprovação dos gates críticos.


In [1]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
from pathlib import Path
import hashlib
import json
import subprocess
import sys

ROOT = Path("/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7")
SCRIPT_CANDIDATES = [
    Path("/content/SPINE_GPEv7_PNAD_COVID_CERTIFIER_v1.0.2.py"),
    ROOT / "scripts/SPINE_GPEv7_PNAD_COVID_CERTIFIER_v1.0.2.py",
]
REQ_CANDIDATES = [
    Path("/content/requirements_SPINE_GPEv7_PNAD_COVID_CERTIFIER_v1.0.2.txt"),
    ROOT / "scripts/requirements_SPINE_GPEv7_PNAD_COVID_CERTIFIER_v1.0.2.txt",
]
SCRIPT = next((p for p in SCRIPT_CANDIDATES if p.exists()), SCRIPT_CANDIDATES[0])
REQ = next((p for p in REQ_CANDIDATES if p.exists()), REQ_CANDIDATES[0])

print("ROOT:", ROOT)
print("SCRIPT:", SCRIPT, SCRIPT.exists())
print("REQ:", REQ, REQ.exists())
if not SCRIPT.exists():
    raise FileNotFoundError(f"Script não encontrado: {SCRIPT_CANDIDATES}")
if not REQ.exists():
    raise FileNotFoundError(f"Requirements não encontrado: {REQ_CANDIDATES}")


ROOT: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7
SCRIPT: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/scripts/SPINE_GPEv7_PNAD_COVID_CERTIFIER_v1.0.2.py True
REQ: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/scripts/requirements_SPINE_GPEv7_PNAD_COVID_CERTIFIER_v1.0.2.txt True


In [3]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--prefer-binary", "-r", str(REQ)],
    check=True,
)
subprocess.run([sys.executable, "-m", "py_compile", str(SCRIPT)], check=True)
print("py_compile: OK")


py_compile: OK


## 1. Auditoria da série completa


In [4]:
audit = subprocess.run(
    [
        sys.executable,
        str(SCRIPT),
        "--root", str(ROOT),
        "--mode", "audit",
        "--months", "all",
        "--download",
        "--strict",
    ],
    text=True,
    capture_output=True,
    check=False,
)
print(audit.stdout)
print(audit.stderr)
print("Audit exit code:", audit.returncode)
if audit.returncode != 0:
    raise RuntimeError("A auditoria falhou; revise STDOUT/STDERR antes da certificação.")


2026-07-21 01:04:54,405 | INFO | SPINE-GPE PNAD COVID Certifier v1.0.2 | root=/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7 | mode=audit | months=[5, 6, 7, 8, 9, 10, 11]
2026-07-21 01:05:16,434 | INFO | Auditoria PNAD COVID concluída | status=AUDIT_PASSED


Audit exit code: 0


## 2. Certificação endurecida maio–novembro/2020


In [5]:
cert = subprocess.run(
    [
        sys.executable,
        str(SCRIPT),
        "--root", str(ROOT),
        "--mode", "certify",
        "--months", "all",
        "--download",
        "--chunk-rows", "50000",
        "--strict",
    ],
    text=True,
    capture_output=True,
    check=False,
)
print(cert.stdout)
print(cert.stderr)
print("Certification exit code:", cert.returncode)


2026-07-21 01:06:33,046 | INFO | SPINE-GPE PNAD COVID Certifier v1.0.2 | root=/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7 | mode=certify | months=[5, 6, 7, 8, 9, 10, 11]
2026-07-21 01:06:38,321 | INFO | Extraindo PNAD_COVID_052020.csv -> /content/spine_pnad_covid_cache/PNAD_COVID_052020.csv
2026-07-21 01:06:43,927 | INFO | Mês 05/2020: 50000 registros processados
2026-07-21 01:06:48,140 | INFO | Mês 05/2020: 100000 registros processados
2026-07-21 01:06:52,086 | INFO | Mês 05/2020: 150000 registros processados
2026-07-21 01:06:55,222 | INFO | Mês 05/2020: 200000 registros processados
2026-07-21 01:06:58,369 | INFO | Mês 05/2020: 250000 registros processados
2026-07-21 01:07:02,209 | INFO | Mês 05/2020: 300000 registros processados
2026-07-21 01:07:06,680 | INFO | Mês 05/2020: 349306 registros processados
2026-07-21 01:07:07,593 | INFO | Extraindo PNAD_COVID_062020.csv -> /content/spine_pnad_covid_cache/PNAD_COVID_062020.csv
2026-07-21 01:07:12,450 | INFO | Mês 06/2020: 50000 

## 3. Lock, outputs imutáveis e hashes


In [6]:
LOCK = ROOT / "00_admin/PNAD_COVID_CERTIFICATION_LOCK.json"
if not LOCK.exists():
    raise RuntimeError("O lock não foi criado. Revise a célula de certificação.")

lock = json.loads(LOCK.read_text(encoding="utf-8"))
print(json.dumps(lock, ensure_ascii=False, indent=2))

if lock.get("status") not in {"CERTIFIED", "CORE_CERTIFIED"}:
    raise RuntimeError(
        f"Certificação não liberada: {lock.get('status')}. "
        f"Falhas: {lock.get('critical_failures', [])}"
    )

immutable_output = Path(lock["output"])
latest_alias = Path(lock["output_latest"])
assert immutable_output.exists(), immutable_output
assert latest_alias.exists(), latest_alias
assert lock["run_id"] in immutable_output.name


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(2**20), b""):
            h.update(chunk)
    return h.hexdigest()

assert sha256(immutable_output) == sha256(latest_alias)
assert sha256(immutable_output) == lock["artifact_hashes"]["output"]
print("STATUS:", lock["status"])
print("OUTPUT IMUTÁVEL:", immutable_output)
print("ALIAS LATEST:", latest_alias)
print("SHA-256:", sha256(immutable_output))


{
  "run_id": "20260721T010633Z",
  "script_version": "1.0.2",
  "schema_version": "spine-gpe-v7-pnad-covid-delivery-1.0.1",
  "validation_schema_version": "spine-gpe-v7-pnad-covid-validation-1.0.1",
  "mode": "certify",
  "status": "CERTIFIED",
  "critical_failures": [],
  "secondary_failures": [],
  "warnings": [
    {
      "test_id": "2020m05.C014.coverage",
      "status": "WARN",
      "severity": "high",
      "message": "Cobertura de C014 entre entregadores documentada; o percentual previdenciário usa somente respostas válidas.",
      "month": 5,
      "observed": 65.2241112828439,
      "expected": ">=95% para cobertura ampla",
      "evidence": {
        "n_delivery": 1294,
        "n_valid": 844,
        "n_missing": 450
      }
    },
    {
      "test_id": "2020m06.C014.coverage",
      "status": "WARN",
      "severity": "high",
      "message": "Cobertura de C014 entre entregadores documentada; o percentual previdenciário usa somente respostas válidas.",
      "month": 

## 4. Verificação independente de conta-própria e C014


In [7]:
import pandas as pd

cols = [
    "reference_month",
    "C007",
    "C014",
    "pandemic_delivery_observed",
    "pandemic_delivery_self_employed",
    "social_security_response_valid",
    "social_security_contributor",
]
df = pd.read_parquet(immutable_output, columns=cols)
delivery = df[df["pandemic_delivery_observed"].fillna(False)].copy()

summary = (
    delivery.groupby("reference_month", observed=True)
    .agg(
        n_delivery=("pandemic_delivery_observed", "size"),
        n_self_employed=("pandemic_delivery_self_employed", lambda s: int(s.fillna(False).sum())),
        n_C014_valid=("social_security_response_valid", lambda s: int(s.fillna(False).sum())),
        n_C014_missing=("social_security_response_valid", lambda s: int((~s.fillna(False)).sum())),
    )
    .reset_index()
)
summary["self_employed_percent_unweighted"] = (
    summary["n_self_employed"] / summary["n_delivery"] * 100
)
summary["C014_coverage_percent_unweighted"] = (
    summary["n_C014_valid"] / summary["n_delivery"] * 100
)
print(summary.to_string(index=False))

assert set(delivery["C007"].dropna().astype(str).unique())
assert set(delivery["C014"].dropna().astype(str).unique()) <= {"1", "2"}
assert (summary["n_self_employed"] > 0).all(), "Conta-própria permaneceu zerada."

for row in summary.to_dict("records"):
    audit_row = lock["coverage_audits"][str(int(row["reference_month"]))]
    assert audit_row["n_delivery"] == row["n_delivery"]
    assert audit_row["n_self_employed"] == row["n_self_employed"]
    assert audit_row["n_C014_valid"] == row["n_C014_valid"]
    assert audit_row["n_C014_missing"] == row["n_C014_missing"]

print("Hardening C007/C014: OK")


 reference_month  n_delivery  n_self_employed  n_C014_valid  n_C014_missing  self_employed_percent_unweighted  C014_coverage_percent_unweighted
               5        1294              491           844             450                         37.944359                         65.224111
               6        1381              541           913             468                         39.174511                         66.111513
               7        1401              549           944             457                         39.186296                         67.380443
               8        1456              557           965             491                         38.255495                         66.277473
               9        1443              557           953             490                         38.600139                         66.042966
              10        1451              580           982             469                         39.972433                         67

## 5. Abrir o relatório canônico


In [8]:
report_path = Path(lock["report"])
print("Relatório imutável:", report_path)
print(report_path.read_text(encoding="utf-8")[:12000])


Relatório imutável: /content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/06_reports/pnad_covid_certification/pnad_covid_certification_report_20260721T010633Z.md
# SPINE-GPE v7 — Relatório de Certificação PNAD COVID 2020

- Run ID: `20260721T010633Z`
- Versão: `1.0.2`
- Schema: `spine-gpe-v7-pnad-covid-delivery-1.0.1`
- Validation schema: `spine-gpe-v7-pnad-covid-validation-1.0.1`
- Status: **CERTIFIED**
- Meses: `05,06,07,08,09,10,11`
- Output imutável: `/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/03_processed/20_pnad_covid_certified/certified_pnad_covid_delivery_2020_m05_m11_20260721T010633Z.parquet`
- Alias latest: `/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7/03_processed/20_pnad_covid_certified/certified_pnad_covid_delivery_2020.parquet`

## Limite epistemológico congelado

`pandemic_delivery_observed` é ocupação observada em C007C=16/17. `platform_delivery_direct` permanece NA. O enunciado de C007C=17 menciona exemplos de apps, mas a pesquisa não pergunta diret